# 00 — Revenue Leakage & Customer 360 Lakehouse Setup

## Purpose

This notebook initializes the Unity Catalog objects and landing-zone directories required by the Revenue Leakage & Customer 360 Lakehouse.

The project models a subscription-based business whose customer, subscription, invoice, payment, usage, and support data originates from multiple operational systems.

## Lakehouse Layers

- **Bronze:** Raw and incrementally ingested source data
- **Silver:** Cleaned, validated, standardized, and deduplicated data
- **Gold:** Business-ready dimensions, facts, Customer 360 tables, and revenue-leakage metrics
- **Quarantine:** Records rejected by data-quality rules
- **Monitoring:** Pipeline execution and data-quality metrics

In [0]:
%sql

SELECT
    current_catalog() AS current_catalog,
    current_schema() AS current_schema;

## 1. Create Lakehouse Schemas

Create the schemas used to separate raw ingestion, validated data, business-ready models, rejected records, and operational monitoring.

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS workspace.revenue_leakage_bronze
COMMENT 'Raw and incrementally ingested source data';

CREATE SCHEMA IF NOT EXISTS workspace.revenue_leakage_silver
COMMENT 'Cleaned, validated, standardized, and deduplicated data';

CREATE SCHEMA IF NOT EXISTS workspace.revenue_leakage_gold
COMMENT 'Business-ready dimensions, facts, Customer 360 tables, and revenue-leakage metrics';

CREATE SCHEMA IF NOT EXISTS workspace.revenue_leakage_quarantine
COMMENT 'Invalid records rejected by data-quality rules';

CREATE SCHEMA IF NOT EXISTS workspace.revenue_leakage_monitoring
COMMENT 'Pipeline execution and data-quality metrics';

SHOW SCHEMAS IN workspace LIKE 'revenue_leakage*';

## 2. Create the Landing Volume

Create a Unity Catalog Volume that acts as the landing zone for files arriving from the simulated source systems.

In [0]:
%sql

CREATE VOLUME IF NOT EXISTS workspace.revenue_leakage_bronze.landing
COMMENT 'Landing zone for files arriving from simulated operational source systems';

SHOW VOLUMES IN workspace.revenue_leakage_bronze;

## 3. Create Source-System Directories

Create a separate directory for each operational source system. This structure keeps the raw files organized by source and business entity.

In [0]:
LANDING_PATH = (
    "/Volumes/workspace/"
    "revenue_leakage_bronze/"
    "landing"
)

SOURCE_DATASETS = [
    "crm/customers",
    "subscription_system/subscriptions",
    "billing_system/invoices",
    "payment_gateway/payments",
    "usage_platform/events",
    "support_system/tickets"
]

created_paths = []

for source_dataset in SOURCE_DATASETS:
    dataset_path = f"{LANDING_PATH}/{source_dataset}"
    dbutils.fs.mkdirs(dataset_path)

    created_paths.append(
        (source_dataset, dataset_path)
    )

paths_df = spark.createDataFrame(
    created_paths,
    ["source_dataset", "volume_path"]
)

display(
    paths_df.orderBy("source_dataset")
)